# Gravity Spy glitch classifier — ResNet-34

Classifies LIGO glitch spectrograms (22 classes) with an ImageNet-pretrained ResNet-34.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Method in one line: take the plot area of each of the 4 spectrogram durations (0.5/1/2/4 s),
convert each to greyscale, stack them as a 4-channel image, fine-tune ResNet-34 on that.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, re, glob, random
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torchvision
from torchvision.transforms import v2 as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 224                         # ResNet's native resolution
DURATIONS = ["0.5", "1.0", "2.0", "4.0"]  # the 4 time windows Gravity Spy renders per glitch
EPOCHS = 25
BATCH = 64
LR = 1e-3
print("device:", DEVICE, "| torch", torch.__version__)

## 1. Download the data
Uses the public Kaggle copy of the Gravity Spy training set (already split into train / validation / test).
If the download asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's 🔑 Secrets panel
(get them from kaggle.com → Settings → Create New Token) and re-run.

In [ ]:
import kagglehub
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError:
    pass

DATA_DIR = kagglehub.dataset_download("tentotheminus9/gravity-spy-gravitational-waves")
print(DATA_DIR)

## 2. Index the files
Each glitch is saved as 4 PNGs (one per duration). We group them by glitch ID, take the class from the
folder name and the split from the path.

In [ ]:
name_re = re.compile(r"^(.*?)[_-]?(0\.5|1\.0|2\.0|4\.0)\.png$", re.I)

def split_of(path):
    for part in reversed(path.lower().split(os.sep)):
        if part in ("train", "training"): return "train"
        if part in ("val", "valid", "validation"): return "val"
        if part == "test": return "test"
    return None

groups = defaultdict(dict)  # (split, class, glitch_id) -> {duration: path}
for path in glob.glob(os.path.join(DATA_DIR, "**", "*.png"), recursive=True):
    m = name_re.match(os.path.basename(path))
    if m:
        cls = os.path.basename(os.path.dirname(path))
        groups[(split_of(path), cls, m.group(1))][m.group(2)] = path

# keep glitches that have all 4 durations: (split, class, [4 paths])
samples = [[s, c, [d[k] for k in DURATIONS]] for (s, c, _), d in groups.items() if all(k in d for k in DURATIONS)]
CLASSES = sorted({c for _, c, _ in samples})

# If the folders don't encode a split, make a stratified 70/15/15 one
if any(s not in ("train", "val", "test") for s, _, _ in samples):
    print("No split found in folder names -> making a stratified 70/15/15 split")
    idx, ys = np.arange(len(samples)), [c for _, c, _ in samples]
    tr, rest = train_test_split(idx, test_size=0.3, stratify=ys, random_state=SEED)
    va, te = train_test_split(rest, test_size=0.5, stratify=[ys[i] for i in rest], random_state=SEED)
    for ids, name in ((tr, "train"), (va, "val"), (te, "test")):
        for i in ids: samples[i][0] = name

print(f"{len(samples)} glitches, {len(CLASSES)} classes")
print(Counter(s for s, _, _ in samples))
for c in CLASSES:
    print(f"  {c:28s} {sum(1 for _, cc, _ in samples if cc == c)}")

## 3. Preprocessing: crop → greyscale → stack 4 durations
* The Kaggle PNGs are already just the spectrogram (569×479, no axes), so the auto-crop only trims a 3 px
  border here. It stays as a safeguard for copies of the data that do include axes and a colour bar.
* The viridis colour map increases monotonically in brightness, so greyscale keeps all the information
  while using 1 channel instead of 3.
* The 4 durations become the 4 channels of one image, so the network sees short- and long-timescale
  structure of the same glitch at once.

In [ ]:
def find_plot_box(rgb):
    """(left, top, right, bottom) of the coloured spectrogram area, ignoring white margins, text and colour bar."""
    sat = np.asarray(Image.fromarray(rgb).convert("HSV"))[..., 1] > 80

    def longest_run(frac, thresh=0.4):
        frac = np.convolve(frac, np.ones(9) / 9, mode="same")  # bridge thin grid lines
        best, start = (0, 0), None
        for i, g in enumerate(np.append(frac > thresh, False)):
            if g and start is None: start = i
            if not g and start is not None:
                if i - start > best[1] - best[0]: best = (start, i)
                start = None
        return best

    c0, c1 = longest_run(sat.mean(0))            # widest coloured column block = plot (not the colour bar)
    r0, r1 = longest_run(sat[:, c0:c1].mean(1))
    h, w = sat.shape
    if (c1 - c0) < 0.3 * w or (r1 - r0) < 0.3 * h:  # detection failed -> use whole image
        return (0, 0, w, h)
    m = 3                                         # trim axis lines
    return (c0 + m, r0 + m, c1 - m, r1 - m)

_box_cache = {}
def load_sample(paths):
    chans = []
    for p in paths:
        im = Image.open(p).convert("RGB")
        if im.size not in _box_cache:            # all images share a layout, so detect once per size
            _box_cache[im.size] = find_plot_box(np.asarray(im))
        im = im.crop(_box_cache[im.size]).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        chans.append(np.asarray(im))
    return np.stack(chans)                        # (4, H, W) uint8

# Sanity check: raw image vs. the 4 cropped channels — the crop should contain only the spectrogram
ex = next(s for s in samples if s[0] == "train")
x = load_sample(ex[2])
print("crop box:", _box_cache)
fig, ax = plt.subplots(1, 5, figsize=(18, 3.5))
ax[0].imshow(Image.open(ex[2][1])); ax[0].set_title(f"raw ({ex[1]})")
for i in range(4):
    ax[i + 1].imshow(x[i], cmap="viridis", origin="upper"); ax[i + 1].set_title(f"{DURATIONS[i]} s")
for a in ax: a.axis("off")
plt.show()

In [ ]:
# Load everything into RAM once (≈1.6 GB as uint8) and cache it so re-runs are instant
CACHE = f"/content/gravityspy_{IMG_SIZE}.npz"
if os.path.exists(CACHE):
    d = np.load(CACHE)
    data = {k: (d[f"X_{k}"], d[f"y_{k}"]) for k in ("train", "val", "test")}
else:
    data = {}
    for split in ("train", "val", "test"):
        items = [s for s in samples if s[0] == split]
        with ThreadPoolExecutor(16) as ex_:
            X = np.stack(list(ex_.map(lambda s: load_sample(s[2]), items)))
        y = np.array([CLASSES.index(c) for _, c, _ in items])
        data[split] = (X, y)
        print(split, X.shape)
    np.savez(CACHE, **{f"X_{k}": v[0] for k, v in data.items()}, **{f"y_{k}": v[1] for k, v in data.items()})

## 4. Data loaders and augmentation
Only **time shifts** (±10 % horizontally). No flips: flipping in frequency or time would turn e.g. a
rising *Chirp* into something that isn't one, and the frequency of a glitch is part of its identity
(e.g. *Power_Line* at 60 Hz, *Violin_Mode* at ~500 Hz), so no vertical shifts either.

In [ ]:
class GlitchDS(torch.utils.data.Dataset):
    def __init__(self, X, y, train):
        self.X, self.y, self.train = X, torch.as_tensor(y), train
        self.aug = T.RandomAffine(degrees=0, translate=(0.1, 0))  # (horizontal, vertical) fraction
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i])
        return (self.aug(x) if self.train else x), self.y[i]

def loader(split, train):
    return torch.utils.data.DataLoader(GlitchDS(*data[split], train), batch_size=BATCH, shuffle=train,
                                       drop_last=train, num_workers=2, pin_memory=True)
train_dl, val_dl, test_dl = loader("train", True), loader("val", False), loader("test", False)

MEAN, STD = 0.449, 0.226  # ImageNet statistics averaged over RGB
def prep(x):
    x = x.to(DEVICE, non_blocking=True).float().div_(255).sub_(MEAN).div_(STD)
    return x.contiguous(memory_format=torch.channels_last)

## 5. Model: pretrained ResNet-34 with a 4-channel input
ImageNet weights give the network good edge/texture detectors to start from, which matters with only a
few thousand training images. The first conv layer expects 3 channels; we replace it with a 4-channel one
initialised from the average of the pretrained RGB filters (scaled by 3/4 so activations keep the same size).
The final layer is replaced with a 22-way classifier.

In [ ]:
def build_model(n_classes):
    m = torchvision.models.resnet34(weights="IMAGENET1K_V1")
    w = m.conv1.weight.data                                    # (64, 3, 7, 7)
    m.conv1 = nn.Conv2d(len(DURATIONS), 64, 7, 2, 3, bias=False)
    m.conv1.weight.data = w.mean(1, keepdim=True).repeat(1, len(DURATIONS), 1, 1) * 3 / len(DURATIONS)
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

model = build_model(len(CLASSES)).to(DEVICE).to(memory_format=torch.channels_last)

## 6. Train
* **AdamW + one-cycle LR schedule**: short warm-up then smooth decay — reliable for fine-tuning with no tuning.
* **Label smoothing 0.1**: some Gravity Spy labels are noisy (citizen-science), this stops over-confidence.
* **Mixed precision** (float16) roughly doubles speed on the T4.
* We keep the weights from the epoch with the best **validation** accuracy; the test set is touched only once at the end.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.15)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def predict(dl):
    model.eval()
    preds, labels = [], []
    for x, y in dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            preds.append(model(prep(x)).argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

history, best_acc = [], 0.0
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x, y in train_dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = crit(model(prep(x)), y.to(DEVICE))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    p, l = predict(val_dl)
    acc = (p == l).mean()
    history.append((total / len(train_dl), acc))
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/content/resnet34_gravityspy.pt")
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  train loss {history[-1][0]:.3f}  val acc {acc:.4f}" + ("  *" if acc == best_acc else ""))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([h[0] for h in history]); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot([h[1] for h in history]); ax[1].set_title("validation accuracy"); ax[1].set_xlabel("epoch")
plt.show()

## 7. Evaluate on the held-out test set
Overall accuracy can hide poor performance on the rare classes, so we also print per-class precision/recall
and the macro-F1 (every class weighted equally).

In [ ]:
model.load_state_dict(torch.load("/content/resnet34_gravityspy.pt"))
p, l = predict(test_dl)
print(f"TEST accuracy: {(p == l).mean():.4f}   macro-F1: {f1_score(l, p, average='macro'):.4f}\n")
print(classification_report(l, p, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(l, p, labels=range(len(CLASSES)), normalize="true")
plt.figure(figsize=(11, 9))
plt.imshow(cm, cmap="Blues", vmin=0, vmax=1); plt.colorbar()
plt.xticks(range(len(CLASSES)), CLASSES, rotation=90); plt.yticks(range(len(CLASSES)), CLASSES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Test confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

## 8. Classify a new glitch
Pass the 4 PNG paths (0.5, 1.0, 2.0, 4.0 s) for one glitch.

In [ ]:
@torch.no_grad()
def classify(paths, top=3):
    model.eval()
    x = torch.from_numpy(load_sample(paths))[None]
    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        probs = model(prep(x)).float().softmax(1)[0].cpu()
    return [(CLASSES[i], round(probs[i].item(), 3)) for i in probs.argsort(descending=True)[:top]]

ex = next(s for s in samples if s[0] == "test")
print("true:", ex[1], "| predicted:", classify(ex[2]))